# ModernBERT Masked Language Modeling with OpenVINO

[ModernBERT](https://huggingface.co/blog/modernbert) is an upgraded version of the original BERT architecture, optimized for efficiency and long-context understanding (up to 8k tokens). It features innovations like Rotary Position Embeddings (RoPE), unpadding, and Flash Attention.

This notebook demonstrates how to:
1. Load a pre-trained ModernBERT model from Hugging Face.
2. Convert the model to OpenVINO IR format.
3. Run inference using OpenVINO Runtime.
4. Create an interactive Gradio demo for Masked Language Modeling.

**Note:** This notebook requires `transformers>=4.48.0` for ModernBERT support.


#### Table of contents

- [Imports](#Imports)
- [Load the Model](#load-the-model)
- [Convert Model to OpenVINO IR](#convert-model-to-openvino-ir)
- [Run Inference](#run-inference)
- [Interactive Demo with Gradio](#interactive-demo-with-gradio)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/modernbert-masked-language-modeling/modernbert-masked-language-modeling.ipynb" />

In [9]:
# Install required packages
%pip install -q "openvino>=2024.0.0" "transformers>=4.48.0" "torch>=2.0" "gradio"

Note: you may need to restart the kernel to use updated packages.


## Imports
[back to top ⬆️](#table-of-contents)


In [10]:
import torch
import requests
import numpy as np
import openvino as ov
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Fetch `notebook_utils` module
r = requests.get(
    url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
)

open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("modernbert-masked-language-modeling.ipynb")

## Load the Model
[back to top ⬆️](#table-of-contents)

Load the ModernBERT model and tokenizer from Hugging Face.


In [11]:
# Define model parameters
MODEL_ID = "answerdotai/ModernBERT-base"
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)
OV_MODEL_PATH = MODEL_DIR / "modernbert-masked-lm.xml"

# Load the tokenizer and model from Hugging Face
print(f"Loading {MODEL_ID} from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The base model is pretrained for Masked Language Modeling (MLM).
# We load it using AutoModelForMaskedLM to demonstrate its capability to fill masks.
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
print("Model loaded successfully.")

Loading answerdotai/ModernBERT-base from Hugging Face...


Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Model loaded successfully.


## Convert Model to OpenVINO IR
[back to top ⬆️](#table-of-contents)

We will use `ov.convert_model` to convert the PyTorch model into OpenVINO Intermediate Representation (IR) format. This step optimizes the model for inference on Intel hardware.


In [12]:
if not OV_MODEL_PATH.exists():
    print("Converting model to OpenVINO IR...")

    # Create dummy input for conversion
    dummy_input = tokenizer("The capital of France is [MASK].", return_tensors="pt")

    # Convert the model
    ov_model = ov.convert_model(model, example_input=dict(dummy_input))

    # Save the model
    ov.save_model(ov_model, OV_MODEL_PATH)
    print(f"Model saved to {OV_MODEL_PATH}")
else:
    print(f"Model already exists at {OV_MODEL_PATH}")
    ov_model = ov.Core().read_model(OV_MODEL_PATH)

Model already exists at model/modernbert-masked-lm.xml


## Run Inference
[back to top ⬆️](#table-of-contents)

Now we compile the OpenVINO model and run inference on a sample sentence. You can select the device (CPU, GPU, NPU) for execution.


In [13]:
import ipywidgets as widgets

core = ov.Core()
device = widgets.Dropdown(
    options=core.available_devices + ["AUTO"],
    value="AUTO",
    description="Device:",
    disabled=False,
)

device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

In [14]:
# Compile the model
compiled_model = core.compile_model(ov_model, device_name=device.value)


# Define inference function
def predict_mask(text: str, tokenizer, compiled_model) -> dict:
    """
    Predicts the masked token in the input text.

    :param text: Input sentence with a [MASK] token.
    :param tokenizer: The tokenizer to use for encoding the input.
    :param compiled_model: The OpenVINO compiled model for inference.
    :return: Dictionary containing top 5 predicted tokens and their scores, or an error dictionary.
    """
    if "[MASK]" not in text:
        return {"Error": "Input text must contain [MASK]."}

    inputs = tokenizer(text, return_tensors="np")
    output = compiled_model(dict(inputs))[0]

    # Get the index of the [MASK] token
    mask_token_index = np.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

    # Get logits for the masked token
    mask_token_logits = output[0, mask_token_index, :]

    # Get top 5 predictions
    top_5_tokens = np.argsort(mask_token_logits[0])[-5:][::-1]

    results = {}
    for token in top_5_tokens:
        word = tokenizer.decode([token])
        score = float(mask_token_logits[0, token])
        results[word] = score

    return results


# Test with a sample
sample_text = "The capital of France is [MASK]."
result = predict_mask(sample_text, tokenizer, compiled_model)
print(f"Input: {sample_text}")
print(f"Top predictions: {list(result.keys())}")

Input: The capital of France is [MASK].
Top predictions: [' Paris', ' Lyon', ' Nancy', ' Nice', ' Orleans']


## Interactive Demo with Gradio
[back to top ⬆️](#table-of-contents)

Launch a simple web interface to test the Masked Language Modeling (Fill-Mask) capability of ModernBERT. Type a sentence with `[MASK]` to see the model's predictions.


In [15]:
gradio_helper_path = Path("gradio_helper.py")

if not gradio_helper_path.exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/modernbert-masked-language-modeling/gradio_helper.py",
    )
    gradio_helper_path.open("w").write(r.text)

In [16]:
from gradio_helper import make_demo
from typing import Dict, Union


def gradio_predict(text: str, tokenizer, compiled_model) -> Union[Dict[str, float], str]:
    """
    Predicts the masked token in the input text.

    :param text: Input sentence with a [MASK] token.
    :param tokenizer: The tokenizer to use for encoding the input.
    :param compiled_model: The OpenVINO compiled model for inference.
    :return: Dictionary containing top 5 predicted tokens and their scores, or an error message.
    """
    if "[MASK]" not in text:
        return "Please include [MASK] in your sentence."

    inputs = tokenizer(text, return_tensors="np")
    output = compiled_model(dict(inputs))[0]

    # Get index of [MASK]
    mask_token_indices = np.where(inputs["input_ids"] == tokenizer.mask_token_id)
    if len(mask_token_indices[1]) == 0:
        return "No [MASK] token found."

    mask_idx = mask_token_indices[1][0]

    # Get probabilities
    mask_token_logits = output[0, mask_idx, :]
    probs = torch.softmax(torch.tensor(mask_token_logits), dim=0).numpy()

    # Get top 5
    top_n = 5
    top_indices = np.argsort(probs)[-top_n:][::-1]

    return {tokenizer.decode([idx]): float(probs[idx]) for idx in top_indices}


def predict_fn(text):
    return gradio_predict(text, tokenizer, compiled_model)


demo = make_demo(
    fn=predict_fn,
    title="ModernBERT Masked Language Modeling",
    description="Type a sentence with `[MASK]` to see ModernBERT's predictions (e.g., 'The capital of France is [MASK].'). Running on OpenVINO.",
)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [ ]:
# import shutil
# if MODEL_DIR.exists():
#     shutil.rmtree(MODEL_DIR)